## Câu 1: Thay đổi số lượng epoch

**Yêu cầu:**  
Tăng số lượng epoch từ 5 lên 10 trong phần huấn luyện mô hình.

**Hướng dẫn:**  
Tìm dòng `for epoch in range(5):` và sửa thành `for epoch in range(10):`.  
Sau khi chạy lại mô hình, ghi nhận:

- Độ chính xác trên tập test có thay đổi hay không.
- Biểu đồ loss thay đổi như thế nào qua 10 epoch.

In [ ]:
for epoch in range(10):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    loss_values.append(epoch_loss)

In [ ]:
plt.figure(figsize=(12,5))

# Biểu đồ Loss
plt.subplot(1,2,1)
plt.plot(range(1,11), loss_values, marker='o')
plt.title("Training Loss theo Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Biểu đồ Accuracy
plt.subplot(1,2,2)
plt.plot(range(1,11), accuracy_values, marker='o')
plt.title("Training Accuracy theo Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.show()

**Nhận xét:**

Khi tăng số lượng epoch từ 5 lên 10, mô hình có thêm nhiều lần học trên toàn bộ dữ liệu.  
Do đó giá trị loss giảm dần theo từng epoch và độ chính xác thường tăng lên.  
Tuy nhiên nếu số epoch quá lớn có thể dẫn đến hiện tượng overfitting.

## Câu 2: Thêm một tầng tích chập

**Yêu cầu:**  
Thêm một tầng tích chập thứ ba (`conv3`) vào mô hình `MNIST_CNN`.

**Hướng dẫn:**

- Thêm `self.conv3 = nn.Conv2d(32, 64, kernel_size=3)`
- Thêm lớp này vào hàm forward.
- Sửa kích thước của lớp fully connected thành `64*1*1`.

In [ ]:
class MNIST_CNN(nn.Module):

    def __init__(self):
        super(MNIST_CNN,self).__init__()

        self.conv1 = nn.Conv2d(1,16,3)
        self.conv2 = nn.Conv2d(16,32,3)

        # thêm conv3
        self.conv3 = nn.Conv2d(32,64,3)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(64*1*1,10)

    def forward(self,x):

        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))

        x = self.pool(torch.relu(self.conv3(x)))

        x = x.view(-1,64*1*1)

        x = self.fc1(x)

        return x

Việc thêm tầng tích chập giúp mô hình học được nhiều đặc trưng phức tạp hơn của ảnh.  
Các tầng đầu thường phát hiện cạnh và đường nét, trong khi các tầng sâu hơn phát hiện các mẫu phức tạp hơn.  
Do đó việc tăng số tầng CNN có thể giúp cải thiện độ chính xác của mô hình.

## Câu 3: Thay đổi learning rate

**Yêu cầu:**  
Thử hai giá trị learning rate khác nhau: `0.001` và `0.1`.

**Hướng dẫn:**  
Sửa dòng optimizer:

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

Learning rate quyết định tốc độ cập nhật trọng số của mô hình.  
Nếu learning rate quá nhỏ, mô hình sẽ học rất chậm và mất nhiều thời gian để hội tụ.  
Nếu learning rate quá lớn, quá trình học có thể dao động mạnh và mô hình khó đạt được kết quả tốt.

## Câu 4: Vẽ thêm feature map từ tầng tích chập thứ hai

**Yêu cầu:**  
Sửa hàm `visualize_feature_map` để hiển thị thêm feature map từ tầng `conv2`.

Mục tiêu là quan sát sự khác biệt giữa feature map của các tầng CNN.

In [ ]:
def visualize_feature_map():

    model.eval()

    images,_ = next(iter(test_loader))
    img = images[0:1].to(device)

    conv1_output = torch.relu(model.conv1(img))
    conv2_output = torch.relu(model.conv2(model.pool(conv1_output)))

    img = img.cpu().squeeze()

    plt.figure(figsize=(20,4))

    plt.subplot(1,5,1)
    plt.imshow(img,cmap='gray')
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,5,2)
    plt.imshow(conv1_output[0,0].cpu().detach(),cmap='gray')
    plt.title("Conv1 Map1")
    plt.axis('off')

    plt.subplot(1,5,3)
    plt.imshow(conv1_output[0,1].cpu().detach(),cmap='gray')
    plt.title("Conv1 Map2")
    plt.axis('off')

    plt.subplot(1,5,4)
    plt.imshow(conv2_output[0,0].cpu().detach(),cmap='gray')
    plt.title("Conv2 Map1")
    plt.axis('off')

    plt.subplot(1,5,5)
    plt.imshow(conv2_output[0,1].cpu().detach(),cmap='gray')
    plt.title("Conv2 Map2")
    plt.axis('off')

    plt.show()

Feature map của tầng conv1 thường thể hiện các đặc trưng cơ bản như cạnh và đường nét.  
Trong khi đó feature map của conv2 trừu tượng hơn và tập trung vào các mẫu đặc trưng của chữ số.  
Các tầng sâu hơn trong CNN giúp mô hình nhận diện cấu trúc phức tạp hơn của ảnh.